In [ ]:
%load_ext autoreload 
%autoreload 2
%matplotlib inline
import sys
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
from scipy.stats import gaussian_kde
import json
import os
import re
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import sim_loader as sim_load
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
import matplotlib.lines as mlines 
from matplotlib.animation import FuncAnimation
from matplotlib import pyplot as plt, ticker as mticker
import sim_loader as sim_load
import h5py
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 6))
B_sun = 1e3*u.cm**(-1/2)*u.g**(1/2)/u.s #=1*u.G 1
Bsample = np.array([1e3, 5e3, 1e4])*(u.cm**(-1/2)*u.g**(1/2)/u.s).to(u.M_earth**(1/2)*u.au**(-1/2)/u.Myr)
time = np.linspace(0.1, 5, 100)
for B in Bsample:
    params = code_gas.Params(star_magnetic_field=B)
    print("from code",params.star_magnetic_field)
    print(B)
    mdot_star = M_dot_star(time, params)
    print((mdot_star*u.M_earth/u.Myr).to(u.M_sun/u.yr))
    R_mag = r_magnetic_cavity(mdot_star, params)
    print(R_mag)
    axs.loglog(R_mag,time, linestyle =':', zorder = 0)

axs.set_xlabel('$R_{\\rm mag}$ [AU]', fontsize=25, labelpad=20)
axs.set_ylabel('$t$ [Myr]', fontsize=25, labelpad=20)


In [ ]:
"""
Module with a function for adding the Baraffe et al. (2015)
evolutionary tracks to the database.
"""

from pathlib import Path

import h5py
import numpy as np
import pooch

from beartype import beartype
from species.core import constants



@beartype
def add_baraffe2015(database: h5py._hl.files.File, input_path: str) -> None:
    """
    Function for adding the Baraffe et al. (2015)
    isochrone data to the database.

    Parameters
    ----------
    database : h5py._hl.files.File
        Database.
    input_path : str
        Folder where the data is located.

    Returns
    -------
    NoneType
        None
    """

    url = "http://perso.ens-lyon.fr/isabelle.baraffe/BHAC15dir/BHAC15_tracks+structure"

    iso_tag = "Baraffe et al. (2015)"
    db_tag = "baraffe2015"

    input_file = url.rsplit("/", maxsplit=1)[-1]
    data_file = Path(input_path) / input_file

    if not data_file.exists():
        print()

        pooch.retrieve(
            url=url,
            known_hash="b95474c5d4284373a2fed3f06d969a44bcd925ac0e5b226cc0235acb7e068d2a",
            fname=input_file,
            path=input_path,
            progressbar=True,
        )

    # M/Ms, log t(yr), Teff, log(L/Ls), log(g), R/Rs,
    # Log(Li/Li0), log(Tc), log(ROc), Mrad, Rrad, k2conv, k2rad
    mass, log_age, teff, log_lum, log_g, radius, _, _, _, _, _, _, _ = np.loadtxt(
        data_file, unpack=True, skiprows=45, comments="!"
    )

    age = 1e-6 * 10.0**log_age  # (Myr)
    mass *= constants.M_SUN / constants.M_JUP  # (Msun) -> (Mjup)
    radius *= constants.R_SUN / constants.R_JUP  # (Msun) -> (Mjup)

    iso_data = np.column_stack([age, mass, teff, log_lum, log_g, radius])

    print(f"\nAdding isochrones: {iso_tag}...", end="", flush=True)

    dset = database.create_dataset(
        f"isochrones/{db_tag}/age", data=iso_data[:, 0]
    )  # (Myr)
    database.create_dataset(f"isochrones/{db_tag}/mass", data=iso_data[:, 1])  # (Mjup)
    database.create_dataset(f"isochrones/{db_tag}/teff", data=iso_data[:, 2])  # (K)
    database.create_dataset(
        f"isochrones/{db_tag}/log_lum", data=iso_data[:, 3]
    )  # log(L/Lsun)
    database.create_dataset(f"isochrones/{db_tag}/log_g", data=iso_data[:, 4])  # log(g)
    database.create_dataset(
        f"isochrones/{db_tag}/radius", data=iso_data[:, 5]
    )  # (Rjup)

    dset.attrs["model"] = db_tag

    print(" [DONE]")
    print(f"Database tag: {db_tag}")



In [ ]:
print("Mass range:")
print(R_Baraffe.masses.min(), R_Baraffe.masses.max())

print("\nAge range:")
print(R_Baraffe.log_t_grid.min(), R_Baraffe.log_t_grid.max())

print("\nRequested age range:")
print(np.log10(time.min() * 1e6), np.log10(time.max() * 1e6))

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 6))
params = code_gas.Params()
print(params.star_radius)
params = code_gas.Params(star_radius = "Baraffe")
time = np.linspace(0.52, 5, 100)

for rstar in ["Demircan", "Baraffe"]:
    params = code_gas.Params(star_radius = rstar)
    r_s = R_star(time, params)*u.au.to(u.R_sun)
    radius = np.full_like(time, r_s)
    axs.loglog(time, radius,  linestyle =':', zorder = 0, label = rstar)

axs.set_ylabel('$R_{\\rm star} [R_\\odot]$', fontsize=25, labelpad=20)
axs.set_xlabel('$t$ [Myr]', fontsize=25, labelpad=20)
axs.legend()
plt.savefig("figures/R_star.png", dpi=300, bbox_inches='tight')
fig,axs = plt.subplots(1, 1, figsize=(6, 6))
for rstar in ["Demircan", "Baraffe"]:
    params = code_gas.Params(star_radius = rstar)
    mdot_star = M_dot_star(time, params)
    R_mag = r_magnetic_cavity(mdot_star, time,params)
    axs.loglog(time, R_mag, linestyle =':', zorder = 0, label = rstar)

axs.set_ylabel('$R_{\\rm mag}$ [AU]', fontsize=25, labelpad=20)
axs.set_xlabel('$t$ [Myr]', fontsize=25, labelpad=20)
axs.legend()

fig,axs = plt.subplots(1, 1, figsize=(6, 6))
for lstar in ["Baraffe"]:
    params = code_gas.Params(star_luminosity = lstar)
    axs.loglog(time, L_star(time, params), linestyle =':', zorder = 0, label = lstar)
axs.loglog(time, np.ones_like(time), linestyle =':', zorder = 0, label = "Lsun")
axs.set_ylabel('$L_{\\rm star}$ [L$_\\odot$]', fontsize=25, labelpad=20)
axs.set_xlabel('$t$ [Myr]', fontsize=25, labelpad=20)
axs.legend()
plt.savefig("figures/R_mag.png", dpi=300, bbox_inches='tight')

In [ ]:
# ============================================================
# LOAD SIMULATIONS 
# ============================================================
folder_path_multi = ["sims/new_Rstar"]
folder_path_single = ["sims/opacities/1Msun/Mdot_scatter/singles/kappa_0005","sims/opacities/1Msun/Mdot_scatter/singles/kappa_1"]

simulations_multi = []
sim_parameters_multi = []
parameters_multi = []
simulations_single = []
sim_parameters_single = []
parameters_single = []

for folder in folder_path_multi:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_multi.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_multi.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_multi.append([sim_load.load_params(f) for f in h5_files])

for folder in folder_path_single:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_single.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_single.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_single.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
alpha_transp = 0.2
c = ['orange','teal']

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

for i, folder in enumerate(folder_path_multi):
    for sim, params, sim_params in zip(simulations_multi[i], parameters_multi[i], sim_parameters_multi[i]):
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs[0].scatter(sim.position[p, -1], sim.mass[p, -1], color=c[i])

for i, folder in enumerate(folder_path_single):
    for sim, params, sim_params in zip(simulations_single[i], parameters_single[i], sim_parameters_single[i]):
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs[1].scatter(sim.position[p, -1], sim.mass[p, -1], color=c[i])

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
for i in range(2):
    axs[i].axvline(r_magnetic_cavity(sim_params.t_in, params), linestyle = '-.', color = 'grey', alpha = 0.1)
    axs[i].axvline(r_magnetic_cavity(sim_params.t_fin, params), linestyle = '-.', color = 'grey', alpha = 0.1)
    axs[i].axvspan(r_magnetic_cavity(sim_params.t_in, params), r_magnetic_cavity(sim_params.t_fin, params),facecolor='none', hatch='/', edgecolor='gray', alpha =alpha_transp)
    axs[i].loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
    axs[i].loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
    axs[i].fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   
    plot.boxes(axs[i], 1e-2)
    axs[i].set_xlabel('r [AU]', fontsize=25, labelpad=20)
    axs[i].set_xlim(0.01, 1e2)
    axs[i].set_ylim(1e-3, 1e4)
    axs[i].tick_params(axis="both", which="major", direction='in', size=15, labelsize=18)
    axs[i].tick_params(axis="both", which="minor", direction='in', size=10)
    axs[i].yaxis.set_major_locator(mticker.LogLocator(numticks=999))
    axs[i].yaxis.set_minor_locator(mticker.LogLocator(numticks=999, subs="auto"))
    plot.all_x_ticks(axs[i], num_ticks=100)
    axs[i].set_xscale('log')
    axs[i].set_yscale('log')
    axs[i].axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
kappa_values = [0.005, 1.0]  # List of kappa values corresponding to the folders
# --- legend: one color per kappa value ---
legend_handles = [
    mlines.Line2D([], [], color=c[i], marker='o', linestyle='None', markersize=8,
                  label=fr'$\kappa$ = {kappa_values[i]}m$^2$/kg')
    for i in range(len(folder_path_multi))
]
axs[0].legend(handles=legend_handles, fontsize=14, loc='best')
axs[0].set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)

axs[0].set_title("Multiplanet systems, $N_{\\rm planets} = 5$", fontsize=15, pad=20)
axs[1].set_title("Single planet systems", fontsize=15, pad=20)
plt.savefig("figures/opacities_scatter_multiplanet.png", bbox_inches='tight')
plt.show()